# 🎾 Asistente Experto en Reglamentos de Tenis
### RAG + Agente LangGraph con Google Gemini

**Proyecto Final — IA Generativa** · Máster en Data Science (Evolve)

Este notebook construye un **agente conversacional** que responde dudas sobre el
reglamento del **Ranking Federado del Distrito de Moncloa-Aravaca 2025-2026**
(un torneo de tenis amateur) usando **RAG**. Si la duda es sobre reglas generales
del juego que ese reglamento no cubre, el agente **recurre automáticamente** a las
**Reglas del Tenis de la ITF 2026** como fuente de respaldo.

| Componente | Tecnología |
|---|---|
| LLM y Embeddings | Google Gemini (`gemini-2.5-flash` + `text-embedding-004`) |
| Base vectorial | ChromaDB (2 colecciones, una por reglamento) |
| Framework de agente | LangGraph (`create_react_agent`) |
| Memoria | `InMemorySaver` + `thread_id` |

## 1. Configuración del entorno

Cargamos la API key de Gemini desde `.env` (nunca escrita en el código) y
añadimos la raíz del proyecto al `sys.path` para importar el paquete `src`,
que contiene la lógica reutilizable (misma base de código que la app de Streamlit).

In [ ]:
# Si hace falta, instalá las dependencias (descomentá):
# %pip install -r ../requirements.txt

import sys, pathlib
ROOT = pathlib.Path.cwd().parent   # .../proyecto_IA_gen
sys.path.insert(0, str(ROOT))

from src.config import load_api_key
load_api_key()   # normaliza GEMINI_API_KEY -> GOOGLE_API_KEY
print('API key de Gemini cargada correctamente ✔')

## 2. Construcción de la base de conocimiento vectorial

Cargamos cada PDF, lo troceamos con `RecursiveCharacterTextSplitter`
(1000 caracteres, 150 de solape) y creamos **dos colecciones ChromaDB**
independientes con **Gemini Embeddings**. Mantenerlas separadas es lo que
permite el enrutamiento jerárquico del agente (torneo ▶ ITF).

> `obtener_vectorstores` **construye** las colecciones la primera vez (indexando
> por lotes para respetar el límite de ~100 embeddings/min del free tier de Gemini)
> y en ejecuciones posteriores las **carga de disco** (`./chroma_db`), sin volver a
> gastar cuota. Para forzar una reconstrucción, poné `RECONSTRUIR = True`.

In [ ]:
from src.rag import obtener_vectorstores, construir_vectorstores

RECONSTRUIR = False   # poné True para reindexar desde cero
if RECONSTRUIR:
    vs_torneo, vs_itf = construir_vectorstores(persistir=True)
else:
    vs_torneo, vs_itf = obtener_vectorstores(persistir=True)

print('Colección TORNEO:', vs_torneo._collection.count(), 'fragmentos')
print('Colección ITF   :', vs_itf._collection.count(), 'fragmentos')

### 2.1. Verificación de la base vectorial (antes de conectar el agente)

Como recomienda la consigna, comprobamos que cada colección responde a
consultas de prueba con una búsqueda por similitud directa.

In [ ]:
from src.rag import formatear_documentos

print('--- TORNEO: "puntos del ranking" ---')
print(formatear_documentos(vs_torneo.similarity_search('cómo se reparten los puntos del ranking', k=2)))
print()
print('--- ITF: "tie-break" ---')
print(formatear_documentos(vs_itf.similarity_search('cómo funciona el tie-break', k=2)))

## 3. System prompt y herramientas del agente

El agente dispone de **dos herramientas de recuperación** y un **system prompt**
que impone la jerarquía: primero el reglamento del torneo, y solo si no encuentra
la respuesta, el reglamento ITF. Las decisiones de diseño del prompt están
justificadas en el README. Lo mostramos aquí para dejarlo a la vista:

In [ ]:
from src.agent import SYSTEM_PROMPT
print(SYSTEM_PROMPT)

## 4. Construcción del agente en LangGraph

`create_react_agent` arma el grafo ReAct: recibe la pregunta, decide qué
herramienta usar, recupera contexto de ChromaDB y genera la respuesta con Gemini.
El `checkpointer` (`InMemorySaver`) le da **memoria** por `thread_id`.

In [ ]:
from src.agent import construir_agente, preguntar

# Reutiliza las colecciones ya persistidas en el paso 2.
agente = construir_agente(persistir_chroma=True)
print('Agente listo ✔')

### 4.1. Visualización del grafo

In [ ]:
from IPython.display import Image, display
try:
    display(Image(agente.get_graph().draw_mermaid_png()))
except Exception as e:
    print('(No se pudo renderizar el diagrama:', e, ')')
    print(agente.get_graph().draw_ascii())

## 5. Demostración de la memoria de conversación

Hacemos una pregunta y luego una **segunda pregunta que hace referencia a la
anterior** sin repetir el contexto. El agente mantiene la coherencia gracias a la
memoria del `thread_id`.

**Resultado documentado de una ejecución** (prueba de que la memoria funciona):
> **P1:** *¿qué categorías hay en el torneo?* → El reglamento no define categorías
> por edad/sexo, sino dos clasificaciones (General y Regularidad).
>
> **P2:** *¿y cuántos jugadores hay en cada una?* → El agente entiende que **"cada
> una"** se refiere a esas dos clasificaciones mencionadas antes y responde que el
> número es dinámico (se cubre con nuevos participantes; se completa con *Byes* si es
> impar). Nunca se le repitió el contexto: lo recuperó de la memoria.

In [ ]:
tid = 'demo-memoria'
print('P1:', 'cual es el sistema de puntuacion de un partido?')
print('R1:', preguntar(agente, 'cual es el sistema de puntuacion de un partido?', thread_id=tid))
print()
print('P2 (referencia a la anterior):', '¿y quién saca primero?')
print('R2:', preguntar(agente, '¿y quién saca primero?', thread_id=tid))

## 6. Cinco preguntas de ejemplo documentadas

Cinco consultas que ejercitan la jerarquía: unas se responden con el reglamento
del torneo y otras caen al reglamento ITF.

In [ ]:
preguntas_demo = [
    'cómo se reparten los puntos del ranking del torneo?',      # -> TORNEO
    'qué pasa si un jugador no se presenta a su partido?',       # -> TORNEO
    'cuál es el plazo para disputar cada ronda?',                # -> TORNEO
    'cómo funciona un tie-break?',                               # -> ITF (respaldo)
    'cuántos sets se juegan y cómo se gana un set?',             # -> ITF (respaldo)
]

for i, q in enumerate(preguntas_demo, 1):
    print(f'{"="*80}\n[{i}] {q}\n{"-"*80}')
    print(preguntar(agente, q, thread_id=f'demo-{i}'))
    print()

### 6.1. Respuestas obtenidas (documentadas)

Salida real de una ejecución de las 5 preguntas anteriores. Se conserva aquí como
registro por si la cuota gratuita de la API (≈20 preguntas/día en el free tier de
Gemini) está agotada al re-ejecutar. Nótese cómo **[1]–[3] se resuelven con el
reglamento del torneo** y **[4]–[5] caen al reglamento ITF**, citando siempre la fuente.

**[1] ¿Cómo se reparten los puntos del ranking del torneo?** → *(TORNEO)*
> El ranking se divide en dos clasificaciones (Art. 7.1, pág. 6): la **Clasificación
> General** (según los puestos disputados en cada reto) y la **Clasificación de la
> Regularidad** (mayor número de partidos, sets y juegos ganados). Los nuevos
> participantes entran mediante un *partido de nivel* cuyo resultado asigna un puesto
> (pág. 4). Las penalizaciones por no disputar partidos hacen bajar posiciones (Art. 6.1.1).

**[2] ¿Qué pasa si un jugador no se presenta a su partido?** → *(TORNEO)*
> Se da el partido por perdido por **W.O. (Walk Over)** tras 20 minutos de cortesía.
> Además, quien no comparece debe abonar el importe total de la reserva de pista al
> jugador que sí se presentó, salvo que este pueda aprovecharla (Art. 1.6 y 1.7).

**[3] ¿Cuál es el plazo para disputar cada ronda?** → *(TORNEO)*
> El reglamento fija los plazos de contacto y disputa de cada reto; no cumplirlos
> conlleva penalización en el ranking (Art. 6.1.1).

**[4] ¿Cómo funciona un tie-break?** → *(ITF — respaldo)*
> Gana el primero en llegar a **7 puntos con margen de 2** (si no, continúa hasta
> lograrlo). Saca un punto quien corresponda y luego se alterna cada dos puntos; se
> cambia de lado cada 4 puntos (Reglas ITF 2026, págs. 9 y 32).

**[5] ¿Cuántos sets se juegan y cómo se gana un set?** → *(ITF — respaldo)*
> Un set se gana al llegar a **6 juegos con diferencia de 2**; con 6-6 se juega
> tie-break. El partido se disputa al mejor de 3 o 5 sets según la modalidad (Reglas ITF 2026).

## 7. Celda interactiva de chat

Ejecutá esta celda y escribí tus preguntas. Escribí `salir` para terminar.
Comparte un mismo `thread_id`, así que la conversación tiene memoria.

In [ ]:
tid_chat = 'chat-interactivo'
print('🎾 Asistente del reglamento (escribí "salir" para terminar)\n')
while True:
    try:
        pregunta = input('Vos > ').strip()
    except (EOFError, KeyboardInterrupt):
        break
    if pregunta.lower() in {'salir', 'exit', 'quit', ''}:
        print('¡Hasta luego!')
        break
    print('Asistente >', preguntar(agente, pregunta, thread_id=tid_chat), '\n')